In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

T = "../results/tables/"
df = pd.read_csv("../data/processed/dataset_v2.csv", dtype=str, keep_default_na=False)
assert set(df["split"]) == {"train", "val", "test"}
df["y"] = (df["final_label"] == "attack").astype(int)
df["short_text"] = df["text"].str[:3000]
train_df = df[df["split"] == "train"]
val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)
print("train:", len(train_df), "| val:", len(val_df), "| test:", len(test_df))

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=200000)
clf = LogisticRegression(max_iter=1000)
clf.fit(vec.fit_transform(train_df["short_text"]), train_df["y"])

saved = pd.read_csv(T + "tfidf_lr_v2_val_probs.csv").set_index("id")["prob_attack"]
val_now = clf.predict_proba(vec.transform(val_df["short_text"]))[:, 1]
diff = np.abs(val_now - saved.loc[val_df["id"]].values)
print(f"refit check on all validation prompts: mean abs difference {diff.mean():.6f} | max {diff.max():.6f}")
assert diff.max() < 1e-3, "The refitted TF-IDF model does not match the one evaluated on validation. Do not run the test."

test_probs = clf.predict_proba(vec.transform(test_df["short_text"]))[:, 1]
out = test_df[["id", "source_dataset", "y"]].copy()
out["prob_attack"] = test_probs
out.to_csv(T + "tfidf_lr_v2_test_probs.csv", index=False)
print("TF-IDF | test prompts scored:", len(out), "| all probabilities finite:", bool(np.isfinite(test_probs).all()))

train: 17413 | val: 3298 | test: 3300
refit check on all validation prompts: mean abs difference 0.000000 | max 0.000000
TF-IDF | test prompts scored: 3300 | all probabilities finite: True


In [2]:
import torch
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("mps")
MODEL_DIR = "../models/distilbert_v2"
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()


def score(texts, batch_size=64):
    id_lists = [tok(t[:4000], truncation=True, max_length=256)["input_ids"] for t in texts]
    probs = []
    with torch.no_grad():
        for i in range(0, len(id_lists), batch_size):
            seqs = [torch.tensor(x) for x in id_lists[i:i + batch_size]]
            lengths = torch.tensor([len(s) for s in seqs])
            ids = pad_sequence(seqs, batch_first=True, padding_value=tok.pad_token_id)
            mask = (torch.arange(ids.shape[1])[None, :] < lengths[:, None]).long()
            logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            probs.extend(torch.softmax(logits, dim=-1)[:, 1].cpu().tolist())
    return np.array(probs)


saved = pd.read_csv(T + "distilbert_v2_val_probs.csv").set_index("id")["prob_attack"]
sub = val_df.iloc[:200]
diff = np.abs(score(sub["text"]) - saved.loc[sub["id"]].values)
print(f"loading check on 200 validation prompts: mean abs difference {diff.mean():.4f} | max {diff.max():.4f}")
assert diff.mean() < 0.005 and diff.max() < 0.1, "The DistilBERT checkpoint does not reproduce the saved validation probabilities. Do not run the test."

test_probs = score(test_df["text"])
out = test_df[["id", "source_dataset", "y"]].copy()
out["prob_attack"] = test_probs
out.to_csv(T + "distilbert_v2_test_probs.csv", index=False)
print("DistilBERT | test prompts scored:", len(out), "| all probabilities finite:", bool(np.isfinite(test_probs).all()))

/opt/anaconda3/envs/pg-detector/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1121 17:51:02.662000 1295 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|████████████████████████████████████████████████████████████| 104/104 [00:00<00:00, 22179.67it/s]


loading check on 200 validation prompts: mean abs difference 0.0000 | max 0.0000
DistilBERT | test prompts scored: 3300 | all probabilities finite: True


In [3]:
import os
found = sorted(f for f in os.listdir(T) if f.endswith("_test_probs.csv"))
print(found)
assert len(found) == 4, "Expected four test-probability files"

['distilbert_v2_test_probs.csv', 'lora_test_probs.csv', 'qlora_test_probs.csv', 'tfidf_lr_v2_test_probs.csv']
